# Task 4: Transfer Learning Techniques

This notebook immplements the transfer-learning techniques solution for the project, and its outputs are intended to be analysed in conjunction with the other notebooks created for this project. The transfer-learning techniques implemented are:

1. **Domain-Adaptive Pretraining (DAPT)**  
   This process involves pre-training the BERT model on masked and unlabelled domain specific data, after which the model is then further trained on the labelled domain training data. It is then tested on the domain test data.


2. **Few-Shot Fine-Tuning**  
   This process involves training the BERT model on a comparatively smaller set of samples from exclusively the domain specific training dataset. After this, the model is tested against the domain specific test dataset.

These techniques demonstrate how transfer learning helps BERT adapt from a general understanding of language to domain-specific sentiment understanding. This can then ideally be used for sentiment classification.


## Setup 1: Libraries

To train the BERT model using the aforementioned techniques, a number of Python libraries will need to be installed to provide the necessary functions and logic that can facilitate that process. These libraries include:
- **Transformers**: This provides the transformer model architecture and the ability to acquire pre-trained models from the internet, specifically from Hugging Face.
- **PyTorch**: This library provides the necessary functions and classes for the training of these acquired models
- **Scikit-Learn**: This library provides tools for preparing the data, and evaluating model performance
- **Pandas**: Pandas provides useful data structures such as dataframes and series which are integral to implementing this design
- **NumPy**: NumPy provides a range of useful mathematical tools and objects
- **Random**: This library provides tools for random number generation in Python, which will be used for the creation of random seeds

In [ ]:
#Install the aforementioned libraries
!pip install transformers torch scikit-learn pandas numpy

These libraries and their specific functions and objects are then imported for use later on.

In [ ]:
#Import pandas, numpy and random
import pandas as pd
import numpy as np
import random

#Import training/test split function, and scoring functions from scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

#Import classes from transformers, specifically for tokenisation and modelling
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    BertForMaskedLM,
    DataCollatorForLanguageModeling
)

#Import data management tools and the AdamW optimiser
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

/Users/finleyedmonds/TECHNOLOGY-DESIGN-PROJECT/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup 2: Computing Device and Random Seed

### Computing Device

In order to run this code most efficiently, the PyTorch training setup must be commanded to use whatever devices are available on the system. This ensures faster run-times, where certain cells in this program may run for more than 10 minutes at a time.

In [ ]:
#Set MPS as default device
import torch

#Check for MPS or Cuda availability and set device accordingly. Default to CPU otherwise
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

#Set default tensor type for MPS
if device.type == "mps":
    torch.set_default_tensor_type('torch.FloatTensor')

Using MPS device


/Users/finleyedmonds/TECHNOLOGY-DESIGN-PROJECT/.venv/lib/python3.14/site-packages/torch/__init__.py:1323: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/tensor/python_tensor.cpp:436.)
  _C._set_default_tensor_type(t)


### Random Seed

To ensure the reproducability of the results of this learning process, a random seed will be set, but may be randomised for different outcomes.

In [ ]:
#Set random seed
SEED = 0

#Set seeds for functions in the random, numpy, and torch libraries
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

#If using cuda, or MPS, set seed there
if device == "mps":
    torch.mps.seed(SEED)
elif device == "cuda":
    torch.cuda.manual_seed_all(SEED)

## Setup 3: Data Loading and Processing

A number of pre-prepared datasets have been created to ensure comparable results across the project. These must be loaded and checked for consistency, before training can be done. The datasets utilised include:
- "goemotions_5class.csv": This dataset contains all of the general domain labelled data
- "fpb_train.csv": This dataset contains the training split of the domain specific data
- "fpb_test.csv": This dataset contains the testing split of the domain specific data

### Loading the Data

In [ ]:
#Read dataset files into pandas dataframes
general_df = pd.read_csv("../data/processed/goemotions_5class.csv")
domain_train_df = pd.read_csv("../data/processed/fpb_train.csv")
domain_test_df = pd.read_csv("../data/processed/fpb_test.csv")

#Check dataframe shapes
print("General dataset shape:", general_df.shape)
print("Domain train dataset shape:", domain_train_df.shape)
print("Domain test dataset shape:", domain_test_df.shape)

#Display header values from each dataframe
display(general_df.head())
display(domain_train_df.head())
display(domain_test_df.head())

General dataset shape: (43404, 4)
Domain train dataset shape: (3392, 4)
Domain test dataset shape: (727, 4)


,text,label,source,clean_text
0,my favourite food is anything i didn't have to...,Neutral,GoEmotions,my favourite food is anything i didn t have to...
1,"now if he does off himself, everyone will thin...",Neutral,GoEmotions,now if he does off himself everyone will think...
2,why the fuck is bayless isoing,Fear,GoEmotions,why the fuck is bayless isoing
3,to make her feel threatened,Fear,GoEmotions,to make her feel threatened
4,dirty southern wankers,Fear,GoEmotions,dirty southern wankers


,text,label,source,clean_text
0,"in stead of being based on a soft drink , as i...",Neutral,FinancialPhraseBank,in stead of being based on a soft drink as is ...
1,"look out for vintage fabric cushion covers , '...",Neutral,FinancialPhraseBank,look out for vintage fabric cushion covers NUM...
2,"thanks to my nokia and lulu , i am now proud t...",Neutral,FinancialPhraseBank,thanks to my nokia and lulu i am now proud to ...
3,nordstjernan has used its option to buy anothe...,Neutral,FinancialPhraseBank,nordstjernan has used its option to buy anothe...
4,25 november 2010 - finnish paints and coatings...,Neutral,FinancialPhraseBank,NUM november NUM finnish paints and coatings c...


,text,label,source,clean_text
0,the share capital of alma media corporation bu...,Neutral,FinancialPhraseBank,the share capital of alma media corporation bu...
1,the eu commission said earlier it had fined th...,Fear,FinancialPhraseBank,the eu commission said earlier it had fined th...
2,"kesko pursues a strategy of healthy , focused ...",Optimism,FinancialPhraseBank,kesko pursues a strategy of healthy focused gr...
3,down to eur5 .9 m h1 '09 3 august 2009 - finni...,Sadness,FinancialPhraseBank,down to eurNUM NUM m hNUM NUM NUM august NUM f...
4,"cencorp would focus on the development , manuf...",Neutral,FinancialPhraseBank,cencorp would focus on the development manufac...


### Checking Column Names

Text and label column headers must be checked before training can be done, in case any errors have been made in the data loading process.

In [ ]:
#Create lists of all column headers in the datasets
print("General columns:", general_df.columns.tolist())
print("Domain columns:", domain_train_df.columns.tolist())
print("Domain columns:", domain_test_df.columns.tolist())

General columns: ['text', 'label', 'source', 'clean_text']
Domain columns: ['text', 'label', 'source', 'clean_text']
Domain columns: ['text', 'label', 'source', 'clean_text']


In [ ]:
#Set text and label column identifiers based on the above
text_column = "text"
label_column = "label"

### Cleaning the Data

Though the dataset has been preprocessed, redundantly ensuring that it is clean can have no negative outcomes.

In [ ]:
#Drop rows with missing values from the datasets
general_df = general_df[[text_column, label_column]].dropna()
domain_train_df = domain_train_df[[text_column, label_column]].dropna()
domain_test_df = domain_test_df[[text_column, label_column]].dropna()

#Ensure dataset text columns are string type
general_df[text_column] = general_df[text_column].astype(str)
domain_train_df[text_column] = domain_train_df[text_column].astype(str)
domain_test_df[text_column] = domain_test_df[text_column].astype(str)

#Print dataset shapes
print("General dataset after cleaning:", general_df.shape)
print("Domain training dataset after cleaning:", domain_train_df.shape)
print("Domain testing dataset after cleaning:", domain_test_df.shape)

General dataset after cleaning: (43404, 2)
Domain training dataset after cleaning: (3392, 2)
Domain testing dataset after cleaning: (727, 2)


### Label Conversion

The BERT model requires numeric labels for classification, as it cannot interpret string labels. This step has not been performed in the original data processing task, and must be done here. Despite this, the following function has been designed to work whether or not the labels are numeric or not already. It does however employ a hard-coded label mapping to ensure consistency with the Task 3 transformer modelling.

In [ ]:
#Print dataset labels before conversion
print("General labels before conversion:", general_df[label_column].unique())
print("Domain training labels before conversion:", domain_train_df[label_column].unique())
print("Domain testing labels before conversion:", domain_test_df[label_column].unique())

General labels before conversion: <ArrowStringArray>
['Neutral', 'Fear', 'Joy', 'Optimism', 'Sadness']
Length: 5, dtype: str
Domain training labels before conversion: <ArrowStringArray>
['Neutral', 'Optimism', 'Sadness', 'Joy', 'Fear']
Length: 5, dtype: str
Domain testing labels before conversion: <ArrowStringArray>
['Neutral', 'Fear', 'Optimism', 'Sadness', 'Joy']
Length: 5, dtype: str
General dataset final shape: (43404, 2)
Domain training dataset final shape: (3392, 2)
Domain testing dataset final shape: (727, 2)

General label counts:
label
0    17312
1     9302
3     7624
4     6520
2     2646
Name: count, dtype: int64

Domain training label counts:
label
0    2015
1     912
2     391
3      42
4      32
Name: count, dtype: int64

Domain testing label counts:
label
0    432
1    196
2     83
3      9
4      7
Name: count, dtype: int64


In [ ]:
#Create a numeric mapping dictionary for the aforementioned labels. This should be consistent with the Task 3 label mapping dictionary
label_map = {
    "neutral": 0,
    "optimism": 1,
    "sadness": 2,
    "joy": 3,
    "fear": 4
}

#Create a function for converting the labels
def convert_labels(df, label_column):
    #Checks if the label column is numeric
    if pd.api.types.is_numeric_dtype(df[label_column]):
        #Convert to integer if it is
        df[label_column] = df[label_column].astype(int)
        return df

    #Strip leading and following spaces, and ensure the labels are lowercase and string type
    df[label_column] = df[label_column].astype(str).str.lower().str.strip()
    #Map the labels to the label mapping dictionary's associated value
    df[label_column] = df[label_column].map(label_map)
    #Drop rows with missing values in case label mapping has failed
    df = df.dropna()
    #Ensure the label column is integer type
    df[label_column] = df[label_column].astype(int)
    return df

#Convert the datasets' label columns to numeric values
general_df = convert_labels(general_df, label_column)
domain_train_df = convert_labels(domain_train_df, label_column)
domain_test_df = convert_labels(domain_test_df, label_column)

#Print dataset shapes
print("General dataset final shape:", general_df.shape)
print("Domain training dataset final shape:", domain_train_df.shape)
print("Domain testing dataset final shape:", domain_test_df.shape)

#Print label counts for each dataset
print("\nGeneral label counts:")
print(general_df[label_column].value_counts())
print("\nDomain training label counts:")
print(domain_train_df[label_column].value_counts())
print("\nDomain testing label counts:")
print(domain_test_df[label_column].value_counts())

### Tokeniser and Hyperparameters

Now that the data has been cleaned, it needs to be tokenised for input into the BERT pre-trained model. This involves converting the text data into BERT-compatible input IDs and attention masks. First, the model and training hyperparameters will be set, and the computing device will be confirmed.

In [ ]:
#Set BERT tokeniser
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

#Set training hyperparameters
MAX_LEN = 128
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
EPOCHS = 3
NUM_LABELS = 5

#Print device in use
print("Using device:", device)

Using device: mps


Then the dataset will be prepared for use with the BERT model by converting the text data into tokens and returning tensors in the format needed by BERT. This will be done differently based on the domain of the dataset.

In [ ]:
class SentimentDataset(Dataset):
    #Store text, labels, the tokeniser, and the max length hyperparameter
    def __init__(self, texts, labels, tokenizer, max_len=128):
        #Convert input text and labels collection into lists
        self.texts = list(texts)
        self.labels = list(labels)
        
        #Set the tokeniser and max length
        self.tokenizer = tokenizer
        self.max_len = max_len

    #Returns the number of samples in the dataset when called
    def __len__(self):
        return len(self.texts)

    #Retrieves a single sample from the dataset for processing
    def __getitem__(self, index):
        #Get the text retrieved as string type
        text = str(self.texts[index])
        #Get the label retrieved as integer type
        label = int(self.labels[index])

        #Tokenise the text
        encoding = self.tokenizer(
            text, #Input text
            max_length=self.max_len, #Get the maximum token length
            padding="max_length", #Add padding tokens if the sample is under the max length
            truncation=True, #Truncate the tokenised output if over the max length
            return_tensors="pt" #Return pytorch tensors
        )

        #Return a dictionary containing all of the data needed for the BERT training
        return {
            #Provide token ID representations of the text
            "input_ids": encoding["input_ids"].flatten(),
            #Attention mask, where 1 corresponds to a real token and 0 corresponds to a padding token
            "attention_mask": encoding["attention_mask"].flatten(),
            #Set the ground truth sentiment label
            "label": torch.tensor(label, dtype=torch.long)
        }

As DAPT uses unlabelled domain text for masked language modelling, a special class will be created that returns the tokenised text without the associated labels.

In [ ]:
class DomainTextDataset(Dataset):
    #Store text, labels, the tokeniser, and the max length hyperparameter
    def __init__(self, texts, tokenizer, max_len=128):
        #Convert input text collection into lists
        self.texts = list(texts)

        #Set the tokeniser and max length
        self.tokenizer = tokenizer
        self.max_len = max_len

    #Returns the number of samples in the dataset when called
    def __len__(self):
        return len(self.texts)

    #Retrieves a single sample from the dataset for processing
    def __getitem__(self, index):
        #Get the text retrieved as string type
        text = str(self.texts[index])

        #Tokenise the text
        encoding = self.tokenizer(
            text, #Input text
            max_length=self.max_len, #Get the maximum token length
            padding="max_length", #Add padding tokens if the sample is under the max length
            truncation=True, #Truncate the tokenised output if over the max length
            return_special_tokens_mask=True, #Return a mask identifying special tokens
            return_tensors="pt" #Return pytorch tensors
        )

        #Return a dictionary containing all of the data needed for the BERT training
        return {
            #Provide token ID representations of the text
            "input_ids": encoding["input_ids"].flatten(),
            #Attention mask, where 1 corresponds to a real token and 0 corresponds to a padding token
            "attention_mask": encoding["attention_mask"].flatten(),
            #Provide a special token mask. 1 corresponds to special tokens, 0 corresponds to normal tokens
            "special_tokens_mask": encoding["special_tokens_mask"].flatten()
        }

### Training Function

In order to train the model with the specific needs of the project, a bespoke training function must be created.

In [ ]:
#Define a single epoch training cycle
def train_classification_model(model, dataloader, optimizer, device):
    #Put the model intro training mode
    model.train()
    #Pre-set training loss parameter
    total_loss = 0

    #Iterate through all batches in the training dataset
    for batch in dataloader:
        #Clear pytorch gradients from the previous batch as they accumulate between training cycles
        optimizer.zero_grad()

        #Move token ids, attention masks, and labels to the training device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        #Perform a forward pass through the model with these loaded values
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        #Get the training loss for the epoch
        loss = outputs.loss
        #Add this to the total training loss
        total_loss += loss.item()

        #Compute gradients using back propogation
        loss.backward()
        #Update model parameters with the optimiser
        optimizer.step()

    #Return the average training loss across the epochs
    return total_loss / len(dataloader)


### Evaluation Function

With data loaded and a training function in hand, a bespoke evaluation function will be created to capture the necessary comparative statistics for the project. Note that this does not compute the micro F1 scores that the other approaches utilised.

In [ ]:
#Create an evaluation function
def evaluate_model(model, dataloader, device):
    #Set the model into evaluation mode
    model.eval()

    #Create empty lists for the predictions and true labels
    predictions = []
    true_labels = []

    #Disable gradient calculations
    with torch.no_grad():
        #Iterate on all the batches in the data loader
        for batch in dataloader:
            #Send their input ids, attention masks, and labels to the training device
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            #Perform a forward pass through the model with these loaded values
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            #Select the class with the highest predicted probability
            preds = torch.argmax(outputs.logits, dim=1)

            #Move predictions and labels back to the CPU for storage
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    #Compute an accuracy score from the true labels and predictions
    accuracy = accuracy_score(true_labels, predictions)
    
    #Compute precision, recall, and F1 from the true labels and predictions
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    #Print the scores and a classification report
    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)
    print("\nClassification Report:\n")
    print(classification_report(true_labels, predictions, zero_division=0))
    print("\nConfusion Matrix:\n")
    print(confusion_matrix(true_labels, predictions))

    return accuracy, precision, recall, f1


### Train-Test Split

Now that the data has been processed and foundational functions have been established, the datasets can be tokenised for use in training.

In [ ]:
#Tokenise the domain data
domain_train_dataset = SentimentDataset(domain_train_df[text_column], domain_train_df[label_column], tokenizer, MAX_LEN)
domain_test_dataset = SentimentDataset(domain_test_df[text_column], domain_test_df[label_column], tokenizer, MAX_LEN)

#Load the tokenised domain data into the data loader. Shuffle the training data
domain_train_loader = DataLoader(domain_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
domain_test_loader = DataLoader(domain_test_dataset, batch_size=BATCH_SIZE)

#Print the sizes of the training and testing domain datasets
print("Domain training samples:", len(domain_train_dataset))
print("Domain testing samples:", len(domain_test_dataset))

Domain training samples: 3392
Domain testing samples: 727


# Technique 1: Domain-Adaptive Pretraining (DAPT)

DAPT further pretrains BERT on domain-specific text using Masked Language Modelling. DAPT helps BERT learn domain vocabulary and writing patterns before sentiment classification. The pretrained model is then further trained on the labelled domain data, and tested against it as well.

In [ ]:
#Create the DAPT training dataset from the domain training data and tokenise it
dapt_dataset = DomainTextDataset(domain_train_df[text_column].values, tokenizer, MAX_LEN)

#Collate the data for masked language modelling, using a 15% masking probability
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

#Create the DAPT loader using the DAPT training dataset and collator. Shuffle the entries
dapt_loader = DataLoader(
    dapt_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator
)

#Print the size of the DAPT training set
print("DAPT training samples:", len(dapt_dataset))


DAPT training samples: 3392


With the data loaded, the model can now be pre-trained.

In [ ]:
#Acquire the BERT pretraining model for masked language modelling and send it to the computing device
model_dapt_mlm = BertForMaskedLM.from_pretrained("bert-base-uncased")
model_dapt_mlm.to(device)

#Set the optimiser with the pre-specified learning rate hyperparameter
optimizer = AdamW(model_dapt_mlm.parameters(), lr=LEARNING_RATE)

print("Starting Domain-Adaptive Pretraining using Masked Language Modelling")

#Perform one training epoch for MLM
for epoch in range(1):
    #Set the model to training mode
    model_dapt_mlm.train()
    #Preset the total loss parameter to zero
    total_loss = 0

    #Iterate through each batch in the data loader
    for batch in dapt_loader:
        #Reset pytorch gradients to zero
        optimizer.zero_grad()

        #Send input ids, attention masks, and labels to the computing device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        #Perform a forward pass through the model using the loaded data
        outputs = model_dapt_mlm(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        #Increment the total loss using the training loss
        loss = outputs.loss
        total_loss += loss.item()

        #Perform a back propogation pass
        loss.backward()
        #Update the gradients
        optimizer.step()

    #Compute the average loss for the training cycle
    avg_loss = total_loss / len(dapt_loader)
    print(f"DAPT Epoch {epoch + 1}, Loss: {avg_loss}")

#Run time: 2m approx on fin's macbook

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 11474.59it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting Domain-Adaptive Pretraining using Masked Language Modelling
DAPT Epoch 1, Loss: 2.5735514568832687


### Convert DAPT Model to Sentiment Classifier

Now that the model has been pretrained on the unlabelled and masked dataset, it needs to be converted into a sentiment classifier model for final training and evaluation. Model weights and parameters will be carried forward.

In [ ]:
#Set the model to the BERT sequence classifier model using the number of labels hyperparameter
model_dapt_classifier = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=NUM_LABELS
)

#Load the model weights and settings from the MLM model
model_dapt_classifier.bert.load_state_dict(model_dapt_mlm.bert.state_dict(), strict=False)

#Send the classifier model to the computing device
model_dapt_classifier.to(device)

#Set the optimiser with the pre-specified learning rate hyperparameter
optimizer = AdamW(model_dapt_classifier.parameters(), lr=LEARNING_RATE)

print("Fine-tuning DAPT-adapted BERT on labelled domain dataset")

#Fine tune the dataset for the pre-set number of epochs
for epoch in range(EPOCHS):
    print(f"DAPT Classifier Fine-Tuning Epoch {epoch + 1}/{EPOCHS}")
    #Extract the training loss for the epoch and display it
    train_loss = train_classification_model(model_dapt_classifier, domain_train_loader, optimizer, device)
    print("Training Loss:", train_loss)
    #Evaluate the model
    dapt_results = evaluate_model(model_dapt_classifier, domain_test_loader, device)

#Run time: 6m approx in fin's macbook

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9598.06it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

Fine-tuning DAPT-adapted BERT on labelled domain dataset
DAPT Classifier Fine-Tuning Epoch 1/5
Training Loss: 0.6594976220490798
Accuracy: 0.8170563961485557
Precision: 0.7970287981343752
Recall: 0.8170563961485557
F1 Score: 0.8027987137972592

Classification Report:

              precision    recall  f1-score   support

           0       0.83      0.94      0.88       432
           1       0.80      0.65      0.72       196
           2       0.80      0.72      0.76        83
           3       0.00      0.00      0.00         9
           4       0.00      0.00      0.00         7

    accuracy                           0.82       727
   macro avg       0.48      0.46      0.47       727
weighted avg       0.80      0.82      0.80       727


Confusion Matrix:

[[406  20   6   0   0]
 [ 61 128   7   0   0]
 [ 18   5  60   0   0]
 [  1   8   0   0   0]
 [  5   0   2   0   0]]
DAPT Classifier Fine-Tuning Epoch 2/5
Training Loss: 0.3467866161240722
Accuracy: 0.8349381017881705
Preci

With these results, the DAPT training technique has been implemented, and focus can now be put onto the few-shot fine tuning approach.

# Technique 2: Few-Shot Fine-Tuning

Only a small amount of labelled domain data is used for this technique. Few-shot fine-tuning tests whether BERT can adapt when labelled domain data is limited, which is useful for hyper-specialised or data-volume poor fields such as niche sciences or business sectors.

In [ ]:
#Set the proportion of the training dataset to use for the few-shot technique
few_shot_fraction = 0.1

#Create a new train test split utilising only that proportion of the training dataset
few_shot_df, _ = train_test_split(
    pd.DataFrame({
        text_column: domain_train_df[text_column],
        label_column: domain_train_df[label_column]
    }),
    #Set the proportion of the dataset to use
    train_size=few_shot_fraction,
    #Set the random seed for the selection
    random_state=SEED,
    #Ensure dataset classes are represented in equal proportion to the superset
    stratify=domain_train_df[label_column]
)

#Tokenise the data
few_shot_dataset = SentimentDataset(
    few_shot_df[text_column].values,
    few_shot_df[label_column].values,
    tokenizer,
    MAX_LEN
)

#Load the data for training, ensuring it is shuffled
few_shot_loader = DataLoader(few_shot_dataset, batch_size=BATCH_SIZE, shuffle=True)

#Print the label counts and dataset size
print("Few-shot training samples:", len(few_shot_dataset))
print(few_shot_df[label_column].value_counts())


Few-shot training samples: 339
label
0    202
1     91
2     39
3      4
4      3
Name: count, dtype: int64


With the data prepared once again for the few-shot method, the training can now be done.

In [ ]:
#Get the BERT sequence classification model for the set number of labels
model_few_shot = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=NUM_LABELS
)

#Send this model to the computing device
model_few_shot.to(device)

#Set the optimiser for the training with the pre-specified learning rate
optimizer = AdamW(model_few_shot.parameters(), lr=LEARNING_RATE)

#Train the model for the preset number of epochs
for epoch in range(EPOCHS):
    print(f"Few-Shot Fine-Tuning Epoch {epoch + 1}/{EPOCHS}")
    #Extract the training loss for the epoch
    train_loss = train_classification_model(model_few_shot, few_shot_loader, optimizer, device)
    print("Few-Shot Training Loss:", train_loss)
    #Evaluate the model
    few_shot_results = evaluate_model(model_few_shot, domain_test_loader, device)

#Run time: 1m approx in fin's macbook

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9704.97it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

Few-Shot Fine-Tuning Epoch 1/5
Few-Shot Training Loss: 1.2360936890948901
Accuracy: 0.594222833562586
Precision: 0.3531007759271487
Recall: 0.594222833562586
F1 Score: 0.44297543416572416

Classification Report:

              precision    recall  f1-score   support

           0       0.59      1.00      0.75       432
           1       0.00      0.00      0.00       196
           2       0.00      0.00      0.00        83
           3       0.00      0.00      0.00         9
           4       0.00      0.00      0.00         7

    accuracy                           0.59       727
   macro avg       0.12      0.20      0.15       727
weighted avg       0.35      0.59      0.44       727


Confusion Matrix:

[[432   0   0   0   0]
 [196   0   0   0   0]
 [ 83   0   0   0   0]
 [  9   0   0   0   0]
 [  7   0   0   0   0]]
Few-Shot Fine-Tuning Epoch 2/5
Few-Shot Training Loss: 1.0090622901916504
Accuracy: 0.594222833562586
Precision: 0.3531007759271487
Recall: 0.594222833562586
F1 S

With this done, the transfer learning techniques have now been fully implemented and can be prepared for final evaluation.

# Final Transfer Learning Comparison

The model training and testing results can now be collated and converted into a transferable file format for comparison with the other approaches used in the project.

In [ ]:
#Evaluate the DAPT technique again
print("\nFinal Evaluation: DAPT + Domain Fine-Tuning")
dapt_results = evaluate_model(model_dapt_classifier, domain_test_loader, device)

#Evaluate the few-shot technique again
print("\nFinal Evaluation: Few-Shot Fine-Tuning")
few_shot_results = evaluate_model(model_few_shot, domain_test_loader, device)

#Transfer these results into a dataframe
task4_results_df = pd.DataFrame({
    "Transfer Learning Technique": [
        "DAPT + Domain Fine-Tuning",
        "Few-Shot Fine-Tuning"
    ],
    "Accuracy": [
        dapt_results[0],
        few_shot_results[0]
    ],
    "Precision": [
        dapt_results[1],
        few_shot_results[1]
    ],
    "Recall": [
        dapt_results[2],
        few_shot_results[2]
    ],
    "F1 Score": [
        dapt_results[3],
        few_shot_results[3]
    ]
})

#Export this dataset as a .csv file
task4_results_df.to_csv("../dashboard/data/task4_transfer_results_test.csv", index=False)


Final Evaluation: DAPT + Domain Fine-Tuning
Accuracy: 0.7867950481430537
Precision: 0.8032939013512935
Recall: 0.7867950481430537
F1 Score: 0.786615783959302

Classification Report:

              precision    recall  f1-score   support

           0       0.91      0.78      0.84       432
           1       0.64      0.82      0.72       196
           2       0.72      0.90      0.80        83
           3       0.67      0.22      0.33         9
           4       0.00      0.00      0.00         7

    accuracy                           0.79       727
   macro avg       0.59      0.54      0.54       727
weighted avg       0.80      0.79      0.79       727


Confusion Matrix:

[[335  77  19   0   1]
 [ 31 160   4   1   0]
 [  0   7  75   0   1]
 [  1   5   1   2   0]
 [  2   0   5   0   0]]

Final Evaluation: Few-Shot Fine-Tuning
Accuracy: 0.6960110041265475
Precision: 0.6428851032650668
Recall: 0.6960110041265475
F1 Score: 0.6537549859823567

Classification Report:

           